# Snowflake Admin Setup & Credit Analysis

| **Attribute** | **Details** |
|---|---|
| **Author** | Neelakanteswara |
| **Created** | 2025-04-08 |
| **Environment** | Production |
| **Roles Used** | `ACCOUNTADMIN`, `SECURITYADMIN`, `USERADMIN` |

---

**Objectives:**
1. Create dedicated warehouses for full load, incremental load, development, and reporting
2. Configure account timezone and session parameters
3. Create project-based roles with proper permissions
4. Create service and development users and assign default roles and warehouses
5. Tag objects for project-wise cost attribution
6. Set query timeouts and resource monitors
7. Grant object permissions to different roles

---

## Naming Convention

**Pattern:** `<PROJECT>_<OBJECT_NAME>_<SUFFIX>`

All objects end with a **type suffix** for instant identification.

### Account-Level Objects

| **Object** | **Suffix** | **Pattern** | **Examples** |
|---|---|---|---|
| Database | `_DB` | `<PROJECT>_<PURPOSE>_DB` | `PROJECT_1_RAW_DB`, `PROJECT_1_ANALYTICS_DB`, `SHARED_GOVERNANCE_DB` |
| Warehouse | `_WH` | `<PROJECT>_<WORKLOAD>_WH` | `PROJECT_1_DEV_WH`, `PROJECT_1_ETL_WH`, `SHARED_BI_WH` |
| Role | `_RL` | `<PROJECT>_<PURPOSE>_RL` | `PROJECT_1_DEV_RL`, `PROJECT_1_READONLY_RL`, `SHARED_BI_READONLY_RL` |
| Resource Monitor | `_RM` | `<SCOPE>_<PURPOSE>_RM` | `PROJECT_1_DEV_WH_RM`, `ACCOUNT_MONTHLY_RM` |
| User (Person) | — | `<FIRST_LAST>` | `USER_1`, `USER_2` |
| User (Service) | — | `SVC_<PROJECT>_<PURPOSE>` | `SVC_PROJECT_1_ETL`, `SVC_SHARED_BI` |
| Tag | — | `<CATEGORY>` (lowercase) | `project`, `team`, `environment`, `cost_center` |

### Database-Level Objects

| **Object** | **Suffix** | **Pattern** | **Examples** |
|---|---|---|---|
| Schema | `_SCH` | `<PURPOSE>_SCH` | `RAW_SCH`, `STAGING_SCH`, `CURATED_SCH`, `GOVERNANCE_SCH` |
| Table | `_TBL` | `<ENTITY>_TBL` | `CUSTOMERS_TBL`, `ORDERS_TBL`, `DIM_DATE_TBL`, `FACT_SALES_TBL` |
| View | `_VW` | `<ENTITY>_VW` | `ACTIVE_CUSTOMERS_VW`, `MONTHLY_REVENUE_VW` |
| Materialized View | `_MVW` | `<ENTITY>_MVW` | `DAILY_SALES_MVW` |
| Stored Procedure | `_SP` | `<ACTION>_<ENTITY>_SP` | `LOAD_CUSTOMERS_SP`, `MERGE_ORDERS_SP`, `CLEANUP_STALE_DATA_SP` |
| Function (UDF) | `_FN` | `<ACTION>_<ENTITY>_FN` | `MASK_EMAIL_FN`, `CALC_DISCOUNT_FN` |
| Sequence | `_SEQ` | `<ENTITY>_<COLUMN>_SEQ` | `ORDERS_ID_SEQ`, `CUSTOMERS_ID_SEQ` |
| Stage | `_STG` | `<SOURCE>_<PURPOSE>_STG` | `S3_RAW_STG`, `AZURE_LANDING_STG`, `INTERNAL_UPLOAD_STG` |
| File Format | `_FF` | `<TYPE>_<PURPOSE>_FF` | `CSV_DEFAULT_FF`, `JSON_NESTED_FF`, `PARQUET_STANDARD_FF` |
| Pipe | `_PP` | `<SOURCE>_<ENTITY>_PP` | `S3_ORDERS_PP`, `AZURE_CUSTOMERS_PP` |
| Stream | `_SM` | `<ENTITY>_<PURPOSE>_SM` | `ORDERS_CDC_SM`, `CUSTOMERS_CHANGES_SM` |
| Task | `_TSK` | `<ACTION>_<ENTITY>_TSK` | `LOAD_ORDERS_TSK`, `REFRESH_DASHBOARD_TSK`, `CLEANUP_LOGS_TSK` |
| Dynamic Table | `_DT` | `<ENTITY>_<PURPOSE>_DT` | `DAILY_REVENUE_DT`, `CUSTOMER_360_DT` |
| Alert | `_ALT` | `<CONDITION>_ALT` | `HIGH_CREDIT_USAGE_ALT`, `STALE_DATA_ALT` |
| Integration | `_INT` | `<PROVIDER>_<PURPOSE>_INT` | `S3_STORAGE_INT`, `AZURE_NOTIFICATION_INT`, `GCS_LANDING_INT` |
| Network Policy | `_NP` | `<SCOPE>_NP` | `OFFICE_ONLY_NP`, `VPN_ACCESS_NP` |

---

## Notebook Sections

| **Section** | **Contents** |
|---|---|
| **A — Account Configuration** | Warehouses, timezone, roles, users, tags, grants, resource monitors, key-pair auth |
| **B — Credit Analysis** | Total credits, warehouse/user breakdown, daily credits, expensive queries, Cortex Code usage |

---

# Section A: Account Configuration

> This section covers the foundational Snowflake environment setup: warehouses, timezone, roles, users, tags, grants, resource monitors, and key-pair authentication.

---

## Snowflake Role Hierarchy & Best Practices

```
              ACCOUNTADMIN
              /          \
        SECURITYADMIN    SYSADMIN ──── (all custom roles should roll up here)
             |               |
         USERADMIN      PROJECT_1_DEV_RL
                             |
                        PROJECT_1_READONLY_RL
                             |
                        SHARED_BI_READONLY_RL
```

| **Role** | **Purpose** | **Use For** |
|---|---|---|
| `ACCOUNTADMIN` | Top-level role, encapsulates SYSADMIN + SECURITYADMIN | Account-level config, billing, resource monitors, warehouses |
| `SECURITYADMIN` | Holds `MANAGE GRANTS` privilege | Creating roles, granting/revoking privileges on objects |
| `USERADMIN` | Holds `CREATE USER` and `CREATE ROLE` privileges | Creating and managing users |
| `SYSADMIN` | Creates warehouses, databases, and all DB objects | Object creation; all custom roles should roll up here |
| `PUBLIC` | Auto-granted to every user and role | Objects available to everyone |

**Key Best Practices:**
- **Never use `ACCOUNTADMIN` to create objects** — use `SYSADMIN` or custom roles instead
- **Never set `ACCOUNTADMIN` as a user's default role**
- **Grant all custom roles to `SYSADMIN`** so it can manage objects created by those roles
- **Use `SECURITYADMIN`** (not `ACCOUNTADMIN`) for granting privileges
- **Use `USERADMIN`** for creating users and roles
- **Enable MFA** for all users with `ACCOUNTADMIN`

---
## Step 1: Network Policy (Security First)

> **Role:** `SECURITYADMIN` (or `ACCOUNTADMIN`)
>
> **Why first?** Network policies should be created **before** any users or integrations, as they define who can connect to your Snowflake account. This is your first line of defense.
>
> **What it does:** Restricts access to your Snowflake account based on IP addresses. Only IPs in `ALLOWED_IP_LIST` can connect; IPs in `BLOCKED_IP_LIST` are explicitly denied.
>
> **Best Practices:**
> - Create a network policy **before** creating users
> - Always whitelist your office/VPN IP ranges
> - Block known bad IPs or entire ranges
> - Apply at **account level** for global enforcement, or at **user level** for specific restrictions
> - Test with a single user before applying account-wide
>
> **Scope Levels:**
> | Level | Command | Effect |
> |---|---|---|
> | Account | `ALTER ACCOUNT SET NETWORK_POLICY = '...'` | All users must connect from allowed IPs |
> | User | `ALTER USER <user> SET NETWORK_POLICY = '...'` | Only this user is restricted |

In [ ]:
%%sql -r set_role_securityadmin_np
USE ROLE SECURITYADMIN;

In [ ]:
%%sql -r create_network_policy
-- Network policy: Allow only office and VPN IPs
-- CREATE NETWORK POLICY IF NOT EXISTS OFFICE_VPN_NP
--     ALLOWED_IP_LIST = (
--         '203.0.113.0/24',     -- Office IP range
--         '198.51.100.0/24'     -- VPN IP range
--     )
--     BLOCKED_IP_LIST = (
--         '192.0.2.99'          -- Known bad IP
--     )
--     COMMENT = 'Allow office and VPN access only';

-- Apply to entire account (all users)
-- ALTER ACCOUNT SET NETWORK_POLICY = 'OFFICE_VPN_NP';

-- Or apply to a specific user only
-- ALTER USER SVC_PROJECT_1_ETL SET NETWORK_POLICY = 'OFFICE_VPN_NP';

-- Verify active network policies
-- SHOW NETWORK POLICIES;

SELECT 'Network policy commands are commented out — uncomment to enable' AS status;

---
## Step 2: Create Warehouses

> **Role:** `ACCOUNTADMIN`
>
> **Why ACCOUNTADMIN?** Warehouse creation requires the `CREATE WAREHOUSE` privilege, which `SYSADMIN` also has. However, `ACCOUNTADMIN` is used here to also set resource constraints (`STANDARD_GEN_1`). In practice, `SYSADMIN` is preferred for warehouse creation — use `ACCOUNTADMIN` only when account-level settings are needed.
>
> **Best Practices:**
> - Always set `AUTO_SUSPEND` (60s recommended for dev, 300s for ETL)
> - Always set `AUTO_RESUME = TRUE` so warehouses start on demand
> - Use `INITIALLY_SUSPENDED = TRUE` to avoid charges at creation time
> - Size appropriately: `XSMALL` for dev/light ETL, scale up only as needed
> - Separate warehouses by workload type (ETL, dev, reporting) to isolate costs and performance

In [ ]:
%%sql -r set_role_accountadmin_wh
USE ROLE ACCOUNTADMIN;

In [ ]:
%%sql -r create_etl_full_wh
CREATE WAREHOUSE IF NOT EXISTS ETL_FULL_WH
    WAREHOUSE_SIZE = XSMALL
    RESOURCE_CONSTRAINT = STANDARD_GEN_1
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE
    INITIALLY_SUSPENDED = TRUE
    COMMENT = 'Dedicated warehouse for ETL Full Load jobs';

In [ ]:
%%sql -r create_etl_incr_wh
CREATE WAREHOUSE IF NOT EXISTS ETL_INCR_WH
    WAREHOUSE_SIZE = XSMALL
    RESOURCE_CONSTRAINT = STANDARD_GEN_1
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE
    INITIALLY_SUSPENDED = TRUE
    COMMENT = 'Dedicated warehouse for ETL Incremental Load jobs';

In [ ]:
%%sql -r create_dev_wh
CREATE WAREHOUSE IF NOT EXISTS DEV_WH
    WAREHOUSE_SIZE = XSMALL
    RESOURCE_CONSTRAINT = STANDARD_GEN_1
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE
    INITIALLY_SUSPENDED = TRUE
    COMMENT = 'Dedicated warehouse for development purposes';

In [ ]:
%%sql -r create_bi_wh
CREATE WAREHOUSE IF NOT EXISTS BI_WH
    WAREHOUSE_SIZE = XSMALL
    RESOURCE_CONSTRAINT = STANDARD_GEN_1
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE
    INITIALLY_SUSPENDED = TRUE
    COMMENT = 'Dedicated warehouse for BI tool jobs (Power BI, Tableau, etc.)';

In [ ]:
%%sql -r create_medium_wh
CREATE WAREHOUSE IF NOT EXISTS MEDIUM_WH
    WAREHOUSE_SIZE = MEDIUM
    RESOURCE_CONSTRAINT = STANDARD_GEN_1
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE
    INITIALLY_SUSPENDED = TRUE
    COMMENT = 'Dedicated medium size warehouse for data Load jobs';

In [ ]:
%%sql -r create_task_wh
CREATE WAREHOUSE IF NOT EXISTS TASK_WH
    WAREHOUSE_SIZE = XSMALL
    RESOURCE_CONSTRAINT = STANDARD_GEN_1
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE
    INITIALLY_SUSPENDED = TRUE
    COMMENT = 'Dedicated xsmall size warehouse for Task jobs';

---
## Step 3: Configure Timezone

> **Role:** `ACCOUNTADMIN`
>
> **Current Settings:**
> - **Default Timezone:** `America/Los_Angeles`
> - **Region:** `AWS_US_WEST_2`
>
> **Scope Levels:**
> | Level | Command | Effect |
> |---|---|---|
> | Account | `ALTER ACCOUNT SET TIMEZONE = '...'` | Default for all sessions |
> | User | `ALTER USER <user> SET TIMEZONE = '...'` | Override for a specific user |
> | Session | `ALTER SESSION SET TIMEZONE = '...'` | Override for current session only |
>
> **Best Practice:** Set account-level timezone to match your business timezone (e.g., `Asia/Kolkata` for IST). Users can override at session level if needed.

In [ ]:
%%sql -r current_timezone
SHOW PARAMETERS LIKE 'TIMEZONE' IN ACCOUNT;

In [ ]:
%%sql -r account_region_info
SELECT CURRENT_ACCOUNT() AS account, CURRENT_REGION() AS region, CURRENT_TIMESTAMP() AS current_ts;

In [ ]:
%%sql -r timezone_change
-- Change account-level timezone (affects all new sessions)
-- ALTER ACCOUNT SET TIMEZONE = 'Asia/Kolkata';

-- Change timezone for a specific user
-- ALTER USER NEELU SET TIMEZONE = 'Asia/Kolkata';

-- Change timezone for current session only
-- ALTER SESSION SET TIMEZONE = 'Asia/Kolkata';

SELECT 'Timezone commands are commented out — uncomment to change' AS status;

### Query Timeout Settings (Optional)

> Use these to prevent long-running queries from consuming credits. Uncomment and adjust as needed.

In [ ]:
%%sql -r timeout_params
-- ALTER WAREHOUSE DEV_WH SET STATEMENT_TIMEOUT_IN_SECONDS = 300;
-- ALTER SESSION SET STATEMENT_TIMEOUT_IN_SECONDS = 300;

SHOW PARAMETERS LIKE 'STATEMENT_TIMEOUT_IN_SECONDS';

In [ ]:
%%sql -r verify_wh
SHOW WAREHOUSES LIKE 'ETL_FULL_WH';

In [ ]:
%%sql -r grants_wh
SHOW GRANTS ON WAREHOUSE ETL_FULL_WH;

---
## Step 4: Create Roles

> **Role:** `SECURITYADMIN`
>
> **Why SECURITYADMIN?** `SECURITYADMIN` has the `MANAGE GRANTS` privilege, making it the correct role for:
> - Creating custom roles
> - Granting/revoking privileges on objects
> - Managing the role hierarchy
>
> **Note:** `USERADMIN` can also create roles (it has `CREATE ROLE`), but `SECURITYADMIN` is preferred here because the subsequent steps require `MANAGE GRANTS` to assign privileges — avoiding an unnecessary role switch.
>
> **Best Practices:**
> - Follow the principle of least privilege — grant only what's needed
> - Use the naming convention `<PROJECT>_<PURPOSE>_RL`
> - Always add a `COMMENT` explaining the role's purpose
> - Create separate roles for read-only vs read-write access

In [ ]:
%%sql -r set_role_securityadmin_roles
USE ROLE SECURITYADMIN;

In [ ]:
%%sql -r create_roles
CREATE ROLE IF NOT EXISTS PROJECT_1_DEV_RL
    COMMENT = 'Project 1 - Development team role';

CREATE ROLE IF NOT EXISTS PROJECT_1_READONLY_RL
    COMMENT = 'Project 1 - Read-only access role';

CREATE ROLE IF NOT EXISTS SHARED_BI_READONLY_RL
    COMMENT = 'Shared read-only role for BI tools (Power BI, Tableau, etc.)';

---
## Step 5: Grant Custom Roles to SYSADMIN

> **Role:** `SECURITYADMIN`
>
> **Why?** This is a critical best practice. All custom roles **must** be granted to `SYSADMIN` so that:
> - `SYSADMIN` can manage objects created by custom roles
> - `ACCOUNTADMIN` (which inherits from `SYSADMIN`) has full visibility
> - Orphaned objects are avoided — if a custom role is dropped, `SYSADMIN` still has access
>
> **Without this grant**, objects created by the custom role become invisible to `SYSADMIN` and `ACCOUNTADMIN`.

In [ ]:
%%sql -r grant_roles_to_sysadmin
GRANT ROLE PROJECT_1_DEV_RL TO ROLE SYSADMIN;
GRANT ROLE PROJECT_1_READONLY_RL TO ROLE SYSADMIN;
GRANT ROLE SHARED_BI_READONLY_RL TO ROLE SYSADMIN;

---
## Step 6: Create Users

> **Role:** `USERADMIN`
>
> **Why USERADMIN?** `USERADMIN` has the `CREATE USER` and `CREATE ROLE` privileges. It is the designated role for user and role management.
>
> **Note:** While `SECURITYADMIN` inherits all of `USERADMIN`'s privileges (including `CREATE USER` and `CREATE ROLE`), using `USERADMIN` here follows the **least privilege principle** — we only need user creation, not `MANAGE GRANTS`.
>
> **User Types in Snowflake:**
> | Type | Purpose | Auth Method |
> |---|---|---|
> | `PERSON` | Human/interactive users | Password + MFA recommended |
> | `SERVICE` | Applications, ETL pipelines, BI tools | Key-pair auth (no password) |
> | `LEGACY_SERVICE` | Older service accounts | Password-based (deprecated) |
>
> **Best Practices:**
> - Always set `DEFAULT_ROLE` and `DEFAULT_WAREHOUSE` for every user
> - Use `TYPE = SERVICE` for non-interactive accounts (ETL, BI tools)
> - Use `TYPE = PERSON` for human users
> - **Never use `ACCOUNTADMIN` as a default role**
> - Use key-pair authentication for service accounts (no passwords)
> - Enable MFA for all person-type users

In [ ]:
%%sql -r set_role_useradmin
USE ROLE USERADMIN;

In [ ]:
%%sql -r create_user_1
CREATE USER IF NOT EXISTS USER_1
    PASSWORD = 'ChangeMe@123'
    LOGIN_NAME = 'USER_1'
    DISPLAY_NAME = 'User One'
    FIRST_NAME = 'User'
    LAST_NAME = 'One'
    EMAIL = 'user1@company.com'
    DEFAULT_ROLE = PROJECT_1_DEV_RL
    DEFAULT_WAREHOUSE = DEV_WH
    -- DEFAULT_SECONDARY_ROLES = ('ALL')
    MUST_CHANGE_PASSWORD = TRUE
    -- DAYS_TO_EXPIRY = 90
    -- DISABLED = FALSE
    -- MINS_TO_UNLOCK = 30
    -- DEFAULT_NAMESPACE = 'MY_DB.MY_SCHEMA'
    COMMENT = 'Project 1 - Data Engineer'
    TYPE = PERSON;

In [ ]:
%%sql -r create_svc_etl
CREATE USER IF NOT EXISTS SVC_PROJECT_1_ETL
    LOGIN_NAME = 'SVC_PROJECT_1_ETL'
    DISPLAY_NAME = 'Project 1 ETL Service Account'
    EMAIL = 'etl-alerts@company.com'
    DEFAULT_ROLE = PROJECT_1_DEV_RL
    DEFAULT_WAREHOUSE = ETL_FULL_WH
    -- DEFAULT_SECONDARY_ROLES = ('ALL')
    -- DAYS_TO_EXPIRY = 365
    -- DISABLED = FALSE
    -- DEFAULT_NAMESPACE = 'RAW_DB.PUBLIC'
    COMMENT = 'Service user for Project 1 ETL processes'
    TYPE = SERVICE;

In [ ]:
%%sql -r create_svc_bi
CREATE USER IF NOT EXISTS SVC_SHARED_BI
    LOGIN_NAME = 'SVC_SHARED_BI'
    DISPLAY_NAME = 'Shared BI Service Account'
    EMAIL = 'bi-alerts@company.com'
    DEFAULT_ROLE = SHARED_BI_READONLY_RL
    DEFAULT_WAREHOUSE = BI_WH
    -- DEFAULT_SECONDARY_ROLES = ('ALL')
    -- DAYS_TO_EXPIRY = 365
    -- DISABLED = FALSE
    -- DEFAULT_NAMESPACE = 'ANALYTICS_DB.PUBLIC'
    COMMENT = 'Service user for BI tools (Power BI, Tableau, etc.)'
    TYPE = SERVICE;

### MFA Management (Mandatory)

> **Snowflake now enforces MFA for all human users (`TYPE = PERSON`) who authenticate with a password.** This is part of BCR 2086/2097 — single-factor password sign-ins are being deprecated.
>
> **Key facts:**
> - MFA is **mandatory** for all `PERSON` type users on password-based logins
> - `SERVICE` type users are **not affected** (they should use key-pair auth, not passwords)
> - `MFA_ENROLLMENT = OPTIONAL` is deprecated and auto-converts to `REQUIRED_SNOWFLAKE_UI_PASSWORD_ONLY`
> - Snowflake supports **Passkeys**, **TOTP (Authenticator Apps)**, and **Duo** as MFA methods
> - Passkeys are recommended for security and usability
>
> **Recovery commands** (for locked-out users):

In [ ]:
%%sql -r mfa_management
-- Re-enroll a user in MFA (sends email or returns enrollment URL)
ALTER USER USER_1 ENROLL MFA;

-- Temporarily bypass MFA for a locked-out user (30 minutes)
-- ALTER USER USER_1 SET MINS_TO_BYPASS_MFA = 30;

-- Show MFA methods for a user
-- SHOW MFA METHODS FOR USER USER_1;

-- Remove a specific MFA method
-- ALTER USER USER_1 REMOVE MFA METHOD <method_name>;

---
## Step 7: Grant Warehouse Access to Roles

> **Role:** `SECURITYADMIN`
>
> **Why SECURITYADMIN?** Granting privileges on objects is a security operation. `SECURITYADMIN` holds `MANAGE GRANTS`, making it the correct role.
>
> **Best Practices:**
> - Grant `USAGE` (not `OWNERSHIP`) on warehouses to custom roles
> - Match warehouses to roles based on workload: ETL roles get ETL warehouses, reporting roles get reporting warehouses
> - Avoid granting warehouse access to overly broad roles

In [ ]:
%%sql -r set_role_securityadmin_grants
USE ROLE SECURITYADMIN;

In [ ]:
%%sql -r grant_wh_access
GRANT USAGE ON WAREHOUSE ETL_FULL_WH TO ROLE PROJECT_1_DEV_RL;
GRANT USAGE ON WAREHOUSE ETL_INCR_WH TO ROLE PROJECT_1_DEV_RL;
GRANT USAGE ON WAREHOUSE DEV_WH TO ROLE PROJECT_1_DEV_RL;
GRANT USAGE ON WAREHOUSE MEDIUM_WH TO ROLE PROJECT_1_DEV_RL;
GRANT USAGE ON WAREHOUSE TASK_WH TO ROLE PROJECT_1_DEV_RL;

GRANT USAGE ON WAREHOUSE DEV_WH TO ROLE PROJECT_1_READONLY_RL;

GRANT USAGE ON WAREHOUSE BI_WH TO ROLE SHARED_BI_READONLY_RL;

---
## Step 8: Grant Roles to Users

> **Role:** `SECURITYADMIN`
>
> **Best Practices:**
> - Each user should have exactly one `DEFAULT_ROLE`
> - Additional roles can be granted and switched to using `USE ROLE`
> - Avoid granting `ACCOUNTADMIN` widely — only for designated admins
> - Service users should only get the minimum role needed for their workload

In [ ]:
%%sql -r grant_roles_to_users
GRANT ROLE PROJECT_1_DEV_RL TO USER SVC_PROJECT_1_ETL;
GRANT ROLE PROJECT_1_DEV_RL TO USER USER_1;
GRANT ROLE SHARED_BI_READONLY_RL TO USER SVC_SHARED_BI;

-- Grant admin access only to designated admins
-- GRANT ROLE ACCOUNTADMIN TO USER <admin_user>;

---
## Step 9: Grant Account-Level Privileges

> **Role:** `ACCOUNTADMIN`
>
> **Why ACCOUNTADMIN?** Account-level privileges (`CREATE DATABASE`, `CREATE INTEGRATION`, `EXECUTE TASK`) can only be granted by `ACCOUNTADMIN`.
>
> **Best Practices:**
> - Grant `CREATE DATABASE` sparingly — only to roles that genuinely need to create databases
> - Grant `CREATE INTEGRATION` only when the role needs to set up external integrations (S3, Azure, etc.)
> - Grant `EXECUTE TASK` to roles that own scheduled tasks
> - Review these grants periodically to ensure they're still needed

In [ ]:
%%sql -r set_role_accountadmin_acct
USE ROLE ACCOUNTADMIN;

In [ ]:
%%sql -r grant_account_privs
GRANT CREATE DATABASE ON ACCOUNT TO ROLE PROJECT_1_DEV_RL;
GRANT CREATE INTEGRATION ON ACCOUNT TO ROLE PROJECT_1_DEV_RL;
GRANT EXECUTE TASK ON ACCOUNT TO ROLE PROJECT_1_DEV_RL;

---
## Step 10: Resource Monitors

> **Role:** `ACCOUNTADMIN`
>
> **Why ACCOUNTADMIN?** Only `ACCOUNTADMIN` can create and manage resource monitors.
>
> **Scope Levels:**
> - **Account-level:** `ALTER ACCOUNT SET RESOURCE_MONITOR = <monitor_name>` — applies to all warehouses as a safety net
> - **Warehouse-level:** `ALTER WAREHOUSE <wh> SET RESOURCE_MONITOR = <monitor_name>` — granular control per workload
>
> **Best Practices:**
> - Set an **account-level** monitor as a global safety net (e.g., monthly budget cap)
> - Set **warehouse-level** monitors for granular cost control per team/workload
> - Use `NOTIFY` triggers at 50% and 80% for early warning
> - Use `SUSPEND_IMMEDIATE` at 100% to hard-stop spending
> - Set `FREQUENCY` to `MONTHLY` to align with billing cycles
> - Use `NOTIFY_USERS` to alert specific admins via email

In [ ]:
%%sql -r resource_monitors
-- === Warehouse-Level Resource Monitor ===
-- CREATE RESOURCE MONITOR PROJECT_1_DEV_WH_RM
--   WITH CREDIT_QUOTA = 100
--   FREQUENCY = MONTHLY
--   START_TIMESTAMP = IMMEDIATELY
--   -- NOTIFY_USERS = ('USER_1')
--   TRIGGERS
--     ON 50 PERCENT DO NOTIFY
--     ON 80 PERCENT DO NOTIFY
--     ON 100 PERCENT DO SUSPEND_IMMEDIATE;

-- ALTER WAREHOUSE DEV_WH SET RESOURCE_MONITOR = PROJECT_1_DEV_WH_RM;

-- === Account-Level Resource Monitor (global safety net) ===
-- CREATE RESOURCE MONITOR ACCOUNT_MONTHLY_RM
--   WITH CREDIT_QUOTA = 500
--   FREQUENCY = MONTHLY
--   START_TIMESTAMP = IMMEDIATELY
--   -- NOTIFY_USERS = ('ADMIN_USER')
--   TRIGGERS
--     ON 50 PERCENT DO NOTIFY
--     ON 80 PERCENT DO NOTIFY
--     ON 95 PERCENT DO NOTIFY
--     ON 100 PERCENT DO SUSPEND_IMMEDIATE;

-- ALTER ACCOUNT SET RESOURCE_MONITOR = ACCOUNT_MONTHLY_RM;

SELECT 'Resource monitor commands are commented out — uncomment to enable' AS status;

---
## Step 11: Object Tags for Cost Attribution

> **Role:** `ACCOUNTADMIN` (or a role with `CREATE TAG` on the schema and `APPLY TAG` globally)
>
> **Why Tags?** Tags are metadata labels (key-value pairs) attached to Snowflake objects for:
> - **Cost attribution** — track credit consumption by project, team, environment, or cost center
> - **Data governance** — classify sensitive data (PII, etc.)
> - **Object discovery** — find all objects related to a project
>
> **Tags are NOT for access control.** You cannot grant permissions via tags. Use roles for RBAC.
>
> **Tag Inheritance:** Tags set on a database cascade to schemas, tables, and columns. Tags set on a table cascade to all columns.
>
> **Recommended Tags:**
> | Tag | Purpose | Example Values |
> |---|---|---|
> | `project` | Cost attribution by project | `project_1`, `project_2`, `shared` |
> | `team` | Cost attribution by team | `data_engineering`, `analytics`, `ml_ops` |
> | `environment` | Separate dev/staging/prod | `dev`, `staging`, `prod` |
> | `cost_center` | Finance/chargeback | `CC-1001`, `CC-2002` |

In [ ]:
%%sql -r create_tags
CREATE TAG IF NOT EXISTS project
    COMMENT = 'Project name for cost attribution';

CREATE TAG IF NOT EXISTS team
    COMMENT = 'Team name for cost attribution';

CREATE TAG IF NOT EXISTS environment
    COMMENT = 'Environment: dev, staging, prod';

CREATE TAG IF NOT EXISTS cost_center
    COMMENT = 'Cost center code for finance chargeback';

In [ ]:
%%sql -r tag_warehouses
ALTER WAREHOUSE DEV_WH SET TAG
    project = 'project_1', team = 'data_engineering', environment = 'dev';

ALTER WAREHOUSE ETL_FULL_WH SET TAG
    project = 'project_1', team = 'data_engineering', environment = 'prod';

ALTER WAREHOUSE ETL_INCR_WH SET TAG
    project = 'project_1', team = 'data_engineering', environment = 'prod';

ALTER WAREHOUSE BI_WH SET TAG
    project = 'shared', team = 'analytics', environment = 'prod';

ALTER WAREHOUSE TASK_WH SET TAG
    project = 'project_1', team = 'data_engineering', environment = 'prod';

In [ ]:
%%sql -r tag_users
ALTER USER USER_1 SET TAG
    project = 'project_1', team = 'data_engineering';

ALTER USER SVC_PROJECT_1_ETL SET TAG
    project = 'project_1', team = 'data_engineering';

ALTER USER SVC_SHARED_BI SET TAG
    project = 'shared', team = 'analytics';

In [ ]:
%%sql -r verify_tags
SELECT *
FROM TABLE(INFORMATION_SCHEMA.TAG_REFERENCES_ALL_COLUMNS(
    'DEV_WH', 'WAREHOUSE'
));

In [ ]:
%%sql -r credits_by_project_tag
SELECT tag.TAG_VALUE AS project,
       ROUND(SUM(m.credits_used), 2) AS total_credits,
       COUNT(DISTINCT m.warehouse_name) AS warehouses_count
FROM SNOWFLAKE.ACCOUNT_USAGE.TAG_REFERENCES tag
JOIN SNOWFLAKE.ACCOUNT_USAGE.WAREHOUSE_METERING_HISTORY m
    ON tag.OBJECT_NAME = m.WAREHOUSE_NAME
WHERE tag.TAG_NAME = 'PROJECT'
    AND tag.DOMAIN = 'WAREHOUSE'
GROUP BY tag.TAG_VALUE
ORDER BY total_credits DESC;

---
## Step 12: Storage Integrations & External Stages

> **Role:** `ACCOUNTADMIN` (for integration creation), then custom role (for stage creation)
>
> **Flow:**
> 1. **Create Storage Integration** — establishes trust between Snowflake and cloud storage (S3/Azure/GCS)
> 2. **Grant USAGE** on integration to the role that will create stages
> 3. **Create File Formats** — define how to parse CSV/JSON/Parquet files
> 4. **Create External Stage** — points to a specific path in cloud storage using the integration
>
> **Best Practices:**
> - One integration per cloud account/tenant (reuse across stages)
> - Grant `USAGE` on integration only to roles that need it
> - Use IAM roles (S3), Service Principals (Azure), or Service Accounts (GCS) — never raw credentials
> - Always specify `STORAGE_ALLOWED_LOCATIONS` to limit access scope
> - After creation, run `DESCRIBE INTEGRATION <name>` to get the Snowflake-generated IAM user/service principal for trust setup

In [ ]:
%%sql -r set_role_accountadmin_int
USE ROLE ACCOUNTADMIN;

In [ ]:
%%sql -r create_s3_int
-- ═══ Option A: AWS S3 Integration ═══
-- CREATE STORAGE INTEGRATION S3_STORAGE_INT
--     TYPE = EXTERNAL_STAGE
--     STORAGE_PROVIDER = 'S3'
--     ENABLED = TRUE
--     STORAGE_AWS_ROLE_ARN = 'arn:aws:iam::012345678901:role/snowflake-access-role'
--     STORAGE_ALLOWED_LOCATIONS = ('s3://my-bucket/raw/', 's3://my-bucket/staging/')
--     STORAGE_BLOCKED_LOCATIONS = ('s3://my-bucket/secrets/')
--     COMMENT = 'S3 storage integration for raw and staging data';

-- After creation, get IAM user ARN and External ID for S3 trust policy:
-- DESCRIBE INTEGRATION S3_STORAGE_INT;

SELECT 'S3 integration commands are commented out — uncomment to enable' AS status;

In [ ]:
%%sql -r create_azure_int
-- ═══ Option B: Azure Blob Storage Integration ═══
-- CREATE STORAGE INTEGRATION AZURE_STORAGE_INT
--     TYPE = EXTERNAL_STAGE
--     STORAGE_PROVIDER = 'AZURE'
--     ENABLED = TRUE
--     AZURE_TENANT_ID = '<tenant-id>'
--     STORAGE_ALLOWED_LOCATIONS = ('azure://myaccount.blob.core.windows.net/mycontainer/raw/')
--     COMMENT = 'Azure Blob storage integration';

-- DESCRIBE INTEGRATION AZURE_STORAGE_INT;

SELECT 'Azure integration commands are commented out — uncomment to enable' AS status;

In [ ]:
%%sql -r create_gcs_int
-- ═══ Option C: Google Cloud Storage Integration ═══
-- CREATE STORAGE INTEGRATION GCS_STORAGE_INT
--     TYPE = EXTERNAL_STAGE
--     STORAGE_PROVIDER = 'GCS'
--     ENABLED = TRUE
--     STORAGE_ALLOWED_LOCATIONS = ('gcs://my-gcs-bucket/raw/')
--     COMMENT = 'GCS storage integration';

-- DESCRIBE INTEGRATION GCS_STORAGE_INT;

SELECT 'GCS integration commands are commented out — uncomment to enable' AS status;

In [ ]:
%%sql -r grant_int_usage
-- Grant integration USAGE to the role that will create stages
-- GRANT USAGE ON INTEGRATION S3_STORAGE_INT TO ROLE PROJECT_1_DEV_RL;
-- GRANT USAGE ON INTEGRATION AZURE_STORAGE_INT TO ROLE PROJECT_1_DEV_RL;
-- GRANT USAGE ON INTEGRATION GCS_STORAGE_INT TO ROLE PROJECT_1_DEV_RL;

SELECT 'Integration grant commands are commented out — uncomment to enable' AS status;

### Create File Formats & External Stages

> **Role:** `PROJECT_1_DEV_RL` (or any role with `CREATE STAGE` on the schema)
>
> **Two approaches to create external stages:**
>
> | Approach | S3 | Azure | GCS |
> |---|---|---|---|
> | **With Storage Integration** (recommended) | IAM Role | Service Principal | Service Account |
> | **Without Storage Integration** | `AWS_KEY_ID` + `AWS_SECRET_KEY` | `AZURE_SAS_TOKEN` | **Not supported** — GCS requires a storage integration |
>
> **Best Practice:** Always use storage integrations in production. Direct credentials are a security risk and must be manually rotated.
>
> **GCS limitation:** Snowflake has **forbidden** direct credentials for GCS stages. You must use a storage integration.

In [ ]:
%%sql -r create_file_formats
-- USE ROLE PROJECT_1_DEV_RL;
-- USE SCHEMA PROJECT_1_RAW_DB.RAW_SCH;

-- CREATE FILE FORMAT IF NOT EXISTS CSV_DEFAULT_FF
--     TYPE = 'CSV' FIELD_DELIMITER = ',' SKIP_HEADER = 1
--     NULL_IF = ('NULL', 'null', '') EMPTY_FIELD_AS_NULL = TRUE
--     COMPRESSION = AUTO COMMENT = 'Default CSV file format';

-- CREATE FILE FORMAT IF NOT EXISTS JSON_NESTED_FF
--     TYPE = 'JSON' STRIP_OUTER_ARRAY = TRUE
--     COMMENT = 'JSON file format with outer array stripping';

-- CREATE FILE FORMAT IF NOT EXISTS PARQUET_STANDARD_FF
--     TYPE = 'PARQUET' COMPRESSION = SNAPPY
--     COMMENT = 'Standard Parquet file format';

SELECT 'File format commands are commented out — uncomment to enable' AS status;

In [ ]:
%%sql -r create_stages
-- ═══ With Storage Integration (Recommended for Production) ═══

-- CREATE STAGE IF NOT EXISTS S3_RAW_STG
--     STORAGE_INTEGRATION = S3_STORAGE_INT
--     URL = 's3://my-bucket/raw/'
--     DIRECTORY = (ENABLE = TRUE AUTO_REFRESH = TRUE)
--     FILE_FORMAT = CSV_DEFAULT_FF
--     COMMENT = 'S3 external stage using storage integration';

-- CREATE STAGE IF NOT EXISTS AZURE_LANDING_STG
--     STORAGE_INTEGRATION = AZURE_STORAGE_INT
--     URL = 'azure://myaccount.blob.core.windows.net/mycontainer/raw/'
--     DIRECTORY = (ENABLE = TRUE AUTO_REFRESH = TRUE)
--     FILE_FORMAT = CSV_DEFAULT_FF
--     COMMENT = 'Azure external stage using storage integration';

-- CREATE STAGE IF NOT EXISTS GCS_RAW_STG
--     STORAGE_INTEGRATION = GCS_STORAGE_INT
--     URL = 'gcs://my-gcs-bucket/raw/'
--     DIRECTORY = (ENABLE = TRUE AUTO_REFRESH = TRUE)
--     FILE_FORMAT = JSON_NESTED_FF
--     COMMENT = 'GCS external stage using storage integration';

-- ═══ Without Storage Integration (Direct Credentials) ═══

-- S3 with Access Key + Secret Key
-- CREATE STAGE IF NOT EXISTS S3_DEV_STG
--     URL = 's3://my-bucket/dev/'
--     CREDENTIALS = (AWS_KEY_ID = '<access_key>' AWS_SECRET_KEY = '<secret_key>')
--     DIRECTORY = (ENABLE = TRUE)
--     FILE_FORMAT = CSV_DEFAULT_FF
--     COMMENT = 'S3 stage using direct credentials (dev only)';

-- Azure with SAS Token
-- CREATE STAGE IF NOT EXISTS AZURE_DEV_STG
--     URL = 'azure://myaccount.blob.core.windows.net/mycontainer/dev/'
--     CREDENTIALS = (AZURE_SAS_TOKEN = '<sas_token>')
--     DIRECTORY = (ENABLE = TRUE)
--     FILE_FORMAT = CSV_DEFAULT_FF
--     COMMENT = 'Azure stage using SAS token (dev only)';

-- ═══ Internal Stage (Snowflake-managed storage) ═══

-- CREATE STAGE IF NOT EXISTS INTERNAL_UPLOAD_STG
--     DIRECTORY = (ENABLE = TRUE)
--     COMMENT = 'Internal stage for manual file uploads';

-- Manually refresh directory table after file changes
-- ALTER STAGE S3_RAW_STG REFRESH;

-- SHOW STAGES;

SELECT 'Stage commands are commented out — uncomment to enable' AS status;

---
## Step 13: Alerts

> **Role:** `ACCOUNTADMIN` (or a role with `EXECUTE ALERT` privilege)
>
> **What is an Alert?** A scheduled Snowflake object that periodically evaluates a SQL condition. If the condition is met (returns rows), it executes an action (e.g., send email, call a stored procedure).
>
> **Requirements:**
> - A **warehouse** to run the alert's SQL condition check
> - `EXECUTE ALERT` privilege on the account
> - An **email notification integration** (for email alerts)
>
> **Best Practices:**
> - Use alerts for credit spikes, stale data detection, failed tasks, etc.
> - Set `SCHEDULE` to match your monitoring cadence (e.g., every hour, daily)
> - Always `ALTER ALERT ... RESUME` after creation — alerts are created in a suspended state
> - Use `SYSTEM$SEND_EMAIL` for simple email notifications

In [ ]:
%%sql -r set_role_accountadmin_alt
USE ROLE ACCOUNTADMIN;

In [ ]:
%%sql -r create_email_int
-- CREATE NOTIFICATION INTEGRATION IF NOT EXISTS EMAIL_NOTIFICATION_INT
--     TYPE = EMAIL
--     ENABLED = TRUE
--     ALLOWED_RECIPIENTS = ('admin@company.com', 'finance@company.com')
--     COMMENT = 'Email notification integration for alerts';

SELECT 'Notification integration commands are commented out — uncomment to enable' AS status;

In [ ]:
%%sql -r create_credit_alert
-- Credit spike alert: fires if yesterday's credits > 2x the 7-day average
-- CREATE ALERT IF NOT EXISTS HIGH_CREDIT_USAGE_ALT
--     WAREHOUSE = TASK_WH
--     SCHEDULE = 'USING CRON 0 8 * * * UTC'
--     COMMENT = 'Alert when daily credits exceed 2x the 7-day average'
--     IF (EXISTS (
--         SELECT 1
--         FROM SNOWFLAKE.ACCOUNT_USAGE.METERING_DAILY_HISTORY
--         WHERE usage_date = CURRENT_DATE() - 1
--         HAVING SUM(credits_used) > 2 * (
--             SELECT AVG(daily_total) FROM (
--                 SELECT SUM(credits_used) AS daily_total
--                 FROM SNOWFLAKE.ACCOUNT_USAGE.METERING_DAILY_HISTORY
--                 WHERE usage_date BETWEEN CURRENT_DATE() - 8 AND CURRENT_DATE() - 2
--                 GROUP BY usage_date
--             )
--         )
--     ))
--     THEN CALL SYSTEM$SEND_EMAIL(
--         'EMAIL_NOTIFICATION_INT', 'admin@company.com',
--         'Snowflake Credit Spike Alert',
--         'Daily credit usage exceeded 2x the 7-day average. Please review.'
--     );

SELECT 'Credit spike alert is commented out — uncomment to enable' AS status;

In [ ]:
%%sql -r create_stale_alert
-- Stale data alert: fires if no data loaded in 24 hours
-- CREATE ALERT IF NOT EXISTS STALE_DATA_ALT
--     WAREHOUSE = TASK_WH
--     SCHEDULE = 'USING CRON 0 9 * * * UTC'
--     COMMENT = 'Alert when no data loaded in 24 hours'
--     IF (EXISTS (
--         SELECT 1 FROM SNOWFLAKE.ACCOUNT_USAGE.COPY_HISTORY
--         WHERE TABLE_NAME = 'MY_TABLE'
--         HAVING MAX(LAST_LOAD_TIME) < DATEADD(HOUR, -24, CURRENT_TIMESTAMP())
--     ))
--     THEN CALL SYSTEM$SEND_EMAIL(
--         'EMAIL_NOTIFICATION_INT', 'admin@company.com',
--         'Stale Data Alert',
--         'No data loaded into MY_TABLE in the last 24 hours.'
--     );

SELECT 'Stale data alert is commented out — uncomment to enable' AS status;

In [ ]:
%%sql -r resume_alerts
-- IMPORTANT: Alerts are created in SUSPENDED state. Resume to activate:
-- ALTER ALERT HIGH_CREDIT_USAGE_ALT RESUME;
-- ALTER ALERT STALE_DATA_ALT RESUME;
-- SHOW ALERTS;

SELECT 'Alert resume commands are commented out — uncomment to enable' AS status;

---
## Step 14: Review Granted Privileges

> Always verify your grants after setup to confirm everything is correctly configured.

In [ ]:
%%sql -r review_dev_grants
SHOW GRANTS TO ROLE PROJECT_1_DEV_RL;

In [ ]:
%%sql -r review_bi_grants
SHOW GRANTS TO ROLE SHARED_BI_READONLY_RL;

In [ ]:
%%sql -r show_all_roles
SHOW ROLES;

---
## Step 15: Key-Pair Authentication for Service Users

> **Role:** `USERADMIN` (or any role with `OWNERSHIP` on the user)
>
> **Why USERADMIN?** The `ALTER USER ... SET RSA_PUBLIC_KEY` command requires either:
> - `OWNERSHIP` privilege on the user, **or**
> - `MODIFY PROGRAMMATIC AUTHENTICATION METHODS` privilege on the user
>
> `USERADMIN` typically owns the users it creates, so it has `OWNERSHIP` by default.
>
> **Setup Steps:**
> 1. Generate RSA key pair locally: `openssl genrsa 2048 | openssl pkcs8 -topk8 -inform PEM -out rsa_key.p8 -nocrypt`
> 2. Extract public key: `openssl rsa -in rsa_key.p8 -pubout -out rsa_key.pub`
> 3. Set role to `USERADMIN` (which owns the user)
> 4. Paste the public key (without header/footer) in the `ALTER USER` command below
>
> **Key Rotation:** Snowflake supports `RSA_PUBLIC_KEY` and `RSA_PUBLIC_KEY_2` for zero-downtime key rotation

In [ ]:
%%sql -r set_etl_key
USE ROLE USERADMIN;

ALTER USER SVC_PROJECT_1_ETL SET RSA_PUBLIC_KEY = '';

In [ ]:
%%sql -r set_bi_key
ALTER USER SVC_SHARED_BI SET RSA_PUBLIC_KEY = '';

---
## Summary: Role-to-Action Mapping

| **Step** | **Action** | **Correct Role** | **Privilege Required** |
|---|---|---|---|
| 1 | Create network policies | `SECURITYADMIN` | `CREATE NETWORK POLICY` |
| 2 | Create warehouses | `ACCOUNTADMIN` or `SYSADMIN` | `CREATE WAREHOUSE` |
| 3 | Configure timezone | `ACCOUNTADMIN` | Account-level ownership |
| 4 | Create roles | `SECURITYADMIN` or `USERADMIN` | `CREATE ROLE` |
| 5 | Grant roles to SYSADMIN | `SECURITYADMIN` | `MANAGE GRANTS` |
| 6 | Create users | `USERADMIN` | `CREATE USER` |
| 7 | Grant warehouse access | `SECURITYADMIN` | `MANAGE GRANTS` |
| 8 | Grant roles to users | `SECURITYADMIN` | `MANAGE GRANTS` |
| 9 | Grant account-level privileges | `ACCOUNTADMIN` | Account-level ownership |
| 10 | Create resource monitors | `ACCOUNTADMIN` | Account-level ownership |
| 11 | Create/manage tags | `ACCOUNTADMIN` or role with `CREATE TAG` | `CREATE TAG` on schema |
| 12 | Create storage integrations | `ACCOUNTADMIN` | `CREATE INTEGRATION` |
| 12 | Create stages & file formats | Custom role | `USAGE` on integration + `CREATE STAGE` |
| 13 | Create alerts | `ACCOUNTADMIN` | `EXECUTE ALERT` |
| 14 | Review grants | Any role | — |
| 15 | Set key-pair auth | `USERADMIN` | `OWNERSHIP` on user |

---

# Section B: Credit Analysis

> **Role:** `ACCOUNTADMIN`
>
> **Why ACCOUNTADMIN?** All views under `SNOWFLAKE.ACCOUNT_USAGE` (e.g., `METERING_DAILY_HISTORY`, `QUERY_HISTORY`, `QUERY_ATTRIBUTION_HISTORY`, `CORTEX_CODE_SNOWSIGHT_USAGE_HISTORY`) require `IMPORTED PRIVILEGES` on the `SNOWFLAKE` database, which is granted to `ACCOUNTADMIN` by default.
>
> **Important:** Do **not** grant `IMPORTED PRIVILEGES` to broad team roles like `PROJECT_1_DEV_RL` — this would expose sensitive account-level data (query history, user activity, billing) to all users in that role.
>
> **Best Practice:** If you need non-admins to access credit data, create a **dedicated monitoring role** with least-privilege access:
> ```sql
> CREATE ROLE IF NOT EXISTS CREDIT_MONITOR_RL COMMENT = 'Read-only access to account usage for credit monitoring';
> GRANT IMPORTED PRIVILEGES ON DATABASE SNOWFLAKE TO ROLE CREDIT_MONITOR_RL;
> GRANT ROLE CREDIT_MONITOR_RL TO ROLE SYSADMIN;
> GRANT ROLE CREDIT_MONITOR_RL TO USER <finance_user>;
> ```
>
> These queries help you monitor and analyze credit consumption across your Snowflake account.

In [ ]:
%%sql -r set_role_accountadmin_b
USE ROLE ACCOUNTADMIN;

---
## B1: Total Credits Consumed (All Time)

In [ ]:
%%sql -r total_credits_all_time
SELECT
    ROUND(SUM(credits_used), 2) AS total_credits_all_time,
    ROUND(SUM(credits_used_compute), 2) AS compute_credits,
    ROUND(SUM(credits_used_cloud_services), 2) AS cloud_credits,
    ROUND(SUM(credits_billed), 2) AS total_billed,
    MIN(usage_date) AS earliest_date,
    MAX(usage_date) AS latest_date
FROM SNOWFLAKE.ACCOUNT_USAGE.METERING_DAILY_HISTORY;

---
## B2: Credits by Warehouse & by User

> **Note:** These totals will **not** match B1. Here's why:
>
> | View | Covers | Includes Cortex Code / SPCS / Serverless? |
> |---|---|---|
> | `METERING_DAILY_HISTORY` (B1) | All services across the account | Yes — everything |
> | `WAREHOUSE_METERING_HISTORY` (B2a) | Only warehouse compute | No |
> | `QUERY_ATTRIBUTION_HISTORY` (B2b) | Only query-level attributed compute | No (subset of warehouse credits) |
>
> To see the full breakdown including Cortex Code, SPCS, etc., refer to **B1**

In [ ]:
%%sql -r credits_by_warehouse
SELECT
    warehouse_name,
    ROUND(SUM(credits_used), 2) AS total_credits,
    ROUND(SUM(credits_used_compute), 2) AS compute_credits,
    ROUND(SUM(credits_used_cloud_services), 2) AS cloud_credits,
    COUNT(DISTINCT DATE(start_time)) AS active_days
FROM SNOWFLAKE.ACCOUNT_USAGE.WAREHOUSE_METERING_HISTORY
GROUP BY warehouse_name
ORDER BY total_credits DESC;

In [ ]:
%%sql -r credits_by_user
SELECT
    USER_NAME,
    ROUND(SUM(CREDITS_ATTRIBUTED_COMPUTE), 4) AS attributed_compute_credits,
    COUNT(DISTINCT DATE(START_TIME)) AS active_days,
    COUNT(*) AS total_queries
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_ATTRIBUTION_HISTORY
GROUP BY USER_NAME
ORDER BY attributed_compute_credits DESC;

---
## B3: Last Day Credits

In [ ]:
%%sql -r last_day_credits
SELECT
    usage_date, service_type,
    ROUND(credits_used, 4) AS credits_used,
    ROUND(credits_used_compute, 4) AS compute_credits,
    ROUND(credits_used_cloud_services, 4) AS cloud_credits,
    ROUND(credits_billed, 4) AS credits_billed
FROM SNOWFLAKE.ACCOUNT_USAGE.METERING_DAILY_HISTORY
WHERE usage_date = (SELECT MAX(usage_date) FROM SNOWFLAKE.ACCOUNT_USAGE.METERING_DAILY_HISTORY)
ORDER BY credits_used DESC;

---
## B4: Expensive Queries & Long Running Queries

In [ ]:
%%sql -r long_running_queries
SELECT
    QUERY_ID, USER_NAME, WAREHOUSE_NAME, WAREHOUSE_SIZE,
    ROUND(TOTAL_ELAPSED_TIME / 1000, 2) AS total_elapsed_secs,
    ROUND(EXECUTION_TIME / 1000, 2) AS execution_secs,
    ROUND(CREDITS_USED_CLOUD_SERVICES, 6) AS cloud_credits,
    ROWS_PRODUCED, BYTES_SCANNED, QUERY_TYPE,
    SUBSTR(QUERY_TEXT, 1, 200) AS query_preview
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE START_TIME >= DATEADD(DAY, -7, CURRENT_DATE())
  AND EXECUTION_STATUS = 'SUCCESS'
ORDER BY TOTAL_ELAPSED_TIME DESC
LIMIT 20;

In [ ]:
%%sql -r expensive_queries
SELECT
    q.QUERY_ID, q.USER_NAME, q.WAREHOUSE_NAME,
    ROUND(qa.CREDITS_ATTRIBUTED_COMPUTE, 6) AS attributed_credits,
    ROUND(q.TOTAL_ELAPSED_TIME / 1000, 2) AS total_elapsed_secs,
    q.ROWS_PRODUCED, q.BYTES_SCANNED,
    SUBSTR(q.QUERY_TEXT, 1, 200) AS query_preview
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_ATTRIBUTION_HISTORY qa
JOIN SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY q ON qa.QUERY_ID = q.QUERY_ID
WHERE qa.START_TIME >= DATEADD(DAY, -7, CURRENT_DATE())
ORDER BY qa.CREDITS_ATTRIBUTED_COMPUTE DESC
LIMIT 20;

---
## B5: Cortex Code Credits by User

In [ ]:
%%sql -r cortex_code_by_user
SELECT
    u.NAME AS USER_NAME,
    ROUND(SUM(c.TOKEN_CREDITS), 4) AS total_credits,
    SUM(c.TOKENS) AS total_tokens,
    COUNT(*) AS total_requests,
    MIN(c.USAGE_TIME) AS first_usage,
    MAX(c.USAGE_TIME) AS last_usage
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_CODE_SNOWSIGHT_USAGE_HISTORY c
JOIN SNOWFLAKE.ACCOUNT_USAGE.USERS u ON c.USER_ID = u.USER_ID
GROUP BY u.NAME
ORDER BY total_credits DESC;